# Task 3: chẩn đoán → feedback theo level

Bật Internet; không cần GPU vì notebook gọi API. Thêm Kaggle Dataset riêng tư chứa `task3_kaggle_source.zip`, thư mục sample_dataset (32 mẫu), public_test.zip và task3_requested_levels*.json.

`task3_kaggle_source.zip` chứa mã Python, cấu hình và prompt để chạy pipeline trên Kaggle; không chứa model, API key hoặc dữ liệu sinh viên. Cell đầu giải nén mã này vào thư mục làm việc.

Model được cấu hình trong biến `MODEL`: `google/gemma-4-31b-it:free`, gọi qua OpenRouter. Cùng model thực hiện cả chẩn đoán và viết feedback. Đây là ID cấu hình từ notebook cũ của nhóm; cần tài khoản có quyền gọi và model còn được nhà cung cấp phục vụ. Có thể thay MODEL và BASE_URL ở cell cấu hình.

Chỉ 32 mẫu train được dùng làm ví dụ RAG; public test chỉ dùng suy luận. Notebook này không fine-tune, không tải trọng số Qwen và không cần GPU. Fine-tune là thí nghiệm riêng của đồ án.

Chỉ gửi code sinh viên qua API khi được phép theo quy định môn học. Mỗi mẫu cần ít nhất hai lượt gọi; chi phí và quota phụ thuộc nhà cung cấp. Chưa có đánh giá điểm leaderboard cho bản này.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, os, getpass, shutil

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
REPO = WORK / 'LLM-APA-task3'

# Kaggle có thể tự giải nén ZIP khi upload.
sources = sorted({p.parent.parent for p in INPUT.rglob('scripts/predict.py')
                  if (p.parent.parent / 'src/llm_grading').is_dir()
                  and (p.parent.parent / 'tasks/task3_feedback/prompts/v004.txt').is_file()})
archives = list(INPUT.rglob('task3_kaggle_source.zip'))
if len(sources) == 1:
    shutil.copytree(sources[0], REPO, dirs_exist_ok=True)
    print('Đã copy mã nguồn từ:', sources[0])
elif not sources and len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(REPO)
    print('Đã giải nén:', archives[0])
else:
    raise ValueError(f'Cần một bộ mã nguồn duy nhất. Thư mục: {sources}; ZIP: {archives}')

os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyyaml', 'scikit-learn', 'tqdm'], check=True)
sys.path[:0] = [str(REPO), str(REPO / 'src')]

def run(script, *args):
    subprocess.run([sys.executable, str(REPO/'scripts'/script), *map(str,args)], check=True)


In [ ]:
# Nếu tìm thấy nhiều dataset, điền đường dẫn chính xác ở đây.
TRAIN_JSON = None  # '/kaggle/input/.../sample_dataset/task3_feedback.json'
PUBLIC_JSON = None # '/kaggle/input/.../public_test_release/task3_feedback.json'
LEVELS_JSON = None # '/kaggle/input/.../task3_requested_levels....json'

public_archives = list(INPUT.rglob('public_test.zip'))
if PUBLIC_JSON is None and public_archives:
    assert len(public_archives) == 1
    with zipfile.ZipFile(public_archives[0]) as z:
        z.extractall(WORK/'public-data')
files = list(INPUT.rglob('task3_feedback.json')) + list((WORK/'public-data').rglob('task3_feedback.json'))
def choose(training):
    found=[]
    for p in files:
        rows=json.loads(p.read_text(encoding='utf-8-sig'))['samples']
        if bool(rows and 'output' in rows[0]) == training:
            found.append(p)
    assert len(found)==1, f'Điền đường dẫn thủ công, ứng viên: {found}'
    return found[0]
TRAIN_JSON = Path(TRAIN_JSON) if TRAIN_JSON else choose(True)
PUBLIC_JSON = Path(PUBLIC_JSON) if PUBLIC_JSON else choose(False)
if LEVELS_JSON is None:
    matches=list(INPUT.rglob('task3_requested_levels*.json'))
    assert len(matches)==1, 'Cần file requested levels duy nhất'
    LEVELS_JSON=matches[0]
rows=json.loads(Path(LEVELS_JSON).read_text(encoding='utf-8-sig'))
levels={r['sample_id']:r['requested_level'] for r in rows}
assert len(levels)==len(rows) and all(type(v) is int and v in (1,2,3,4) for v in levels.values())
public=json.loads(PUBLIC_JSON.read_text(encoding='utf-8-sig'))['samples']
assert set(levels)=={r['sample_id'] for r in public}
for r in public:
    assert int(r['input']['target_feedback_level'].split()[1]) == levels[r['sample_id']], r['sample_id']
from llm_grading.data.loader import load_samples
train=load_samples(TRAIN_JSON, task='task3')
assert len(train)==32, f'Train phải có 32 mẫu, thấy {len(train)}'
print('Train:', len(train), 'Public:', len(public), 'Level khớp toàn bộ')


In [ ]:
import yaml
# Điền model ID mà tài khoản của bạn có quyền gọi. Có thể đổi endpoint.
MODEL = 'google/gemma-4-31b-it:free'
BASE_URL = 'https://openrouter.ai/api/v1'
KEY_ENV = 'OPENROUTER_API_KEY'
# Kaggle Secrets: thêm OPENROUTER_API_KEY và bật quyền dùng cho notebook.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ[KEY_ENV] = UserSecretsClient().get_secret(KEY_ENV)
except Exception:
    if not os.environ.get(KEY_ENV):
        os.environ[KEY_ENV] = getpass.getpass('API key: ')
OUT = WORK/'task3-v4'
OUT.mkdir(exist_ok=True)
cfg=yaml.safe_load((REPO/'configs/task3/rag.yaml').read_text())
cfg['experiment'].update(id='task3-two-stage-v4', name='task3-two-stage-v4')
cfg['prompt']['version']='v004'
cfg['model']={'provider':'api','name_or_path':MODEL,'base_url':BASE_URL,
    'api_key_env':KEY_ENV,'allow_private_data':True,'timeout_seconds':120,
    'cache_path':str(OUT/'api_cache.jsonl')}
cfg['data'].update(train_path=str(TRAIN_JSON), input_path=str(TRAIN_JSON))
cfg['saving']['output_dir']=str(OUT)
cfg['retrieval'].update(index='bm25',key='code',k=1,compact_examples=True,
    exclude_duplicates=True,exclude_same_exam=False)
cfg['task3'].update(two_stage=True,max_compliance_retries=2,
    fallback_on_failure=False,use_grading_policy=False)
cfg['task2']={'use_grading_policy':False}
cfg['generation'].update(max_new_tokens=1536,max_input_tokens=16384,temperature=0,do_sample=False)
CONFIG=OUT/'config.json'
CONFIG.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
print('Config:',CONFIG)


In [ ]:
# Chạy thử: kiểm tra nhãn, chẩn đoán (responses.jsonl) và feedback.
run('predict.py','--config',CONFIG,'--input',PUBLIC_JSON,
    '--max-samples','3','--output',OUT/'smoke_predictions.json')
print((OUT/'smoke_predictions.json').read_text(encoding='utf-8'))


In [ ]:
# Chạy toàn bộ. Cache tái sử dụng phản hồi đã gọi nếu chạy lại cùng cấu hình.
# Có thể thêm --evidence / --tests nếu đã sinh bằng chứng bằng scripts của repo.
run('predict.py','--config',CONFIG,'--input',PUBLIC_JSON,
    '--output',OUT/'predictions.json')
run('validate_predictions.py','--task','task3','--input',OUT/'predictions.json')
from tasks.task3_feedback.compliance import check_compliance
pred=json.loads((OUT/'predictions.json').read_text(encoding='utf-8'))
assert len(pred)==len(public) and {r['sample_id'] for r in pred}==set(levels)
assert all(r['output']['feedback_level']==levels[r['sample_id']] for r in pred)
assert all(check_compliance(r['output']['feedback'],levels[r['sample_id']])['pass'] for r in pred)
run('package_predictions.py','--task','task3','--input',OUT/'predictions.json',
    '--output',WORK/'submission_task3.zip')
print('Tải /kaggle/working/submission_task3.zip để nộp. Logs/cache chứa dữ liệu riêng tư; không publish notebook output.')
